# Shopping centres in Victoria

This notebook downloads the locations of shopping centres in Victoria
from OpenStreetMap through the Overpass API and saves them in the same
shape as the facilities file: an ID plus `Latitude` and `Longitude`.

## Imports and configuration

The Overpass server asks clients to identify themselves with a
`User-Agent`, so set `USER_AGENT` to something that names you.

In [8]:
import pandas as pd
import requests
import numpy as np
from sklearn.neighbors import BallTree

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
USER_AGENT = "vic-shopping-centres-nikhil-gaba"
SHOPPING_CENTRES_PATH = "shopping_centres.csv"

OVERPASS_QUERY = """
[out:json][timeout:180];
area["ISO3166-2"="AU-VIC"]->.victoria;
nwr["shop"="mall"](area.victoria);
out tags center;
"""

## Ask Overpass for the shopping centres

The query, line by line:

- `[out:json]` asks for JSON, with a 180 second time limit.
- `area["ISO3166-2"="AU-VIC"]->.victoria` finds the Victoria boundary by
  its ISO code, and stores it under the name `victoria`. The code
  avoids matching other places called Victoria.
- `nwr["shop"="mall"](area.victoria)` selects every node, way and
  relation tagged `shop=mall` inside that boundary.
- `out tags center` returns each object's tags and, for ways and
  relations, a centre point.

If zero elements come back, the area lookup failed. Replace that line
with `area["name"="Victoria"]["admin_level"="4"]->.victoria;`. If the
server answers 429 or 504, it is busy, so wait a minute and rerun.

In [5]:
response = requests.post(
    OVERPASS_URL,
    data={"data": OVERPASS_QUERY},
    headers={"User-Agent": USER_AGENT},
    timeout=200,
)
response.raise_for_status()
elements = response.json()["elements"]

print(f"{len(elements)} elements returned")

383 elements returned


## Flatten to a table

Each element becomes one row. Nodes carry `lat` and `lon` at the top
level, while ways and relations carry them inside `center`, so
`element.get("center", element)` picks whichever exists.

The ID combines the OSM type and number (for example `way/12345`) so it
is unique across nodes, ways and relations. The columns are named
`Facility ID`, `Latitude` and `Longitude`, exactly as in the facilities
file, so the routing code can be reused without changes.

In [6]:
def element_to_row(element):
    """Flatten one Overpass element into an ID, name and location.

    Nodes carry their coordinates directly. Ways and relations carry a
    computed centre point instead, under the key "center".
    """
    location = element.get("center", element)
    return {
        "Facility ID": f"{element['type']}/{element['id']}",
        "name": element.get("tags", {}).get("name"),
        "Latitude": location["lat"],
        "Longitude": location["lon"],
    }


shopping_centres = pd.DataFrame(
    [element_to_row(element) for element in elements]
)

## Check and export

Look at the size of the table, missing coordinates, unnamed entries and
repeated names. A mall can appear twice, once as a node and once as its
building outline, so repeated names deserve a look. Then save the table.

In [7]:
n_unnamed = shopping_centres["name"].isna().sum()
n_duplicate_names = shopping_centres["name"].dropna().duplicated().sum()

print(shopping_centres.shape)
print(shopping_centres[["Latitude", "Longitude"]].isna().sum())
print(f"Unnamed: {n_unnamed}")
print(f"Repeated names: {n_duplicate_names}")

shopping_centres.to_csv(SHOPPING_CENTRES_PATH, index=False)
shopping_centres.head(10)

(383, 4)
Latitude     0
Longitude    0
dtype: int64
Unnamed: 36
Repeated names: 36


,Facility ID,name,Latitude,Longitude
0,node/1583426656,Direct Factory Outlets,-37.971080,145.088895
1,node/2960613933,Queen Victoria Market - A Shed,-37.806332,144.957079
2,node/7202000271,North Gateway Shopping Plaza,-37.729179,145.378986
3,node/7715124603,Central Square Shopping Centre,-37.562920,143.855426
4,node/10845639264,Springvale Mega Centre,-37.941372,145.167920
5,node/11834055328,John Thomson & Co,-37.743283,142.024460
6,node/13108363613,None,-38.117456,145.345754
7,node/14137255144,Lalor Hub,-37.664063,145.008007
8,way/4860424,Chadstone Shopping Centre,-37.886409,145.082536
9,way/4965876,Waverley Gardens Shopping Centre,-37.935351,145.190116


## Data Quality & Validation Checks

This section validates the spatial integrity, data formatting, and topological consistency of the shopping centres retrieved from OpenStreetMap. We apply four primary checks:
1. **Data Type & String Standardization:** Enforces `float64` for coordinates, `string` for IDs, and standardizes formatting (e.g., removing extra whitespaces, converting to Title Case).
2. **Coordinate Bounds Validation:** Ensures all points lie strictly within Victoria's geographic bounding box to catch extreme coordinate errors (like Null Island at 0,0).

In [11]:
def clean_basic_data(df):
    """Enforce proper data types and standardize text casing."""
    df = df.copy()
    
    # Enforce numeric and string types
    df['Facility ID'] = df['Facility ID'].astype('string')
    df['Latitude'] = df['Latitude'].astype('float64')
    df['Longitude'] = df['Longitude'].astype('float64')

    # Standardize string formatting for non-null names
    mask = df['name'].notna()
    df.loc[mask, 'name'] = (
        df.loc[mask, 'name']
        .astype(str)
        .str.strip()
        .replace(r'\s+', ' ', regex=True)
        .str.title()
    )
    return df

def check_geographic_bounds(df):
    """Verify that coordinates fall within Victoria's rough bounding box."""
    lat_min, lat_max = -39.2, -33.9
    lon_min, lon_max = 140.9, 150.0

    outliers = df[
        (df['Latitude'] < lat_min) | (df['Latitude'] > lat_max) |
        (df['Longitude'] < lon_min) | (df['Longitude'] > lon_max)
    ]
    return outliers

In [12]:
validated_centres = clean_basic_data(shopping_centres)

print("--- 1. Data Type & String Standardization ---")
print("Data types successfully enforced:")
print(validated_centres.dtypes)
print("\n")

print("--- 2. Coordinate Bounds Validation ---")
outliers = check_geographic_bounds(validated_centres)
if outliers.empty:
    print("All facilities are within Victoria's bounding box.")
else:
    print(f"{len(outliers)} locations found outside Victoria:")
    print(outliers)
print("\n")

--- 1. Data Type & String Standardization ---
Data types successfully enforced:
Facility ID    string[python]
name                   object
Latitude              float64
Longitude             float64
dtype: object


--- 2. Coordinate Bounds Validation ---
All facilities are within Victoria's bounding box.




# Export Final Dataframe to CSV

In [13]:
shopping_centres.to_csv("shopping_centres.csv", index=False)